# RL Test Flow Optimization — Notebook 1 of 3: Setup + Stage-1

**GPU**: Kaggle T4 x2  |  **Runtime target**: <10 hours  |  **Steps**: 1–5

| Notebook | Contents | Est. Time |
|----------|----------|----------|
| **NB1 (this)** | Install, Data, Env, Baselines, Stage-1 (4 algos × 200K) | ~6-8 h |
| NB2 | Stage-2 (top-2 × 3 seeds × 500K) + Optuna (30 trials × 100K) | ~7-9 h |
| NB3 | Final (1M) + Curriculum + All Plots + Export | ~4-5 h |

## Why 3 Notebooks?
- Kaggle GPU sessions time out after **12 hours**
- Full pipeline = ~20+ hours of compute
- Each notebook saves models + results to `/kaggle/working/` for the next

> **After NB1 completes**: Go to Output tab → Create Dataset → name it `rl-stage1-results`
> Then add that dataset as input to NB2.

## Problem: Semiconductor Test Flow Optimization
A chip must pass through 200 tests (voltage, timing, thermal, functional, analog).
Running all tests is expensive. An RL agent learns the **optimal test ordering** to
maximize defect detection while minimizing cost — industry-critical for AMD, Micron, Infineon.

## Algorithms Compared
| Algorithm | Type | Key Feature |
|-----------|------|-------------|
| A2C | On-policy | Fast convergence, synchronous |
| DQN | Off-policy | Experience replay, stable |
| PPO | On-policy | Clipped surrogate objective |
| MaskablePPO | On-policy | Action masking for invalid tests |

> **Production scale note**: This runs 100K chips × 200 tests (Kaggle T4).
> Production: 1M chips × 1000 tests on AMD MI300X / NVIDIA A100.

## Step 1: Environment Setup

In [1]:
import subprocess, sys, os
subprocess.run(['pip', 'install', '-q',
    'stable-baselines3[extra]', 'sb3-contrib', 'gymnasium',
    'optuna', 'mlflow', 'pyarrow', 'matplotlib', 'seaborn',
    'pandas', 'numpy', 'torch'], check=True)

# Always fresh clone → picks up latest code fixes
!rm -rf rl-test-flow-optimization
!git clone https://github.com/rajendarmuddasani/rl-test-flow-optimization.git
os.chdir('rl-test-flow-optimization')
sys.path.insert(0, '.')

import torch
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')
    print(f'VRAM:           {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('No GPU! Go to Settings > Accelerator > GPU T4 x2')

import stable_baselines3 as sb3, sb3_contrib, optuna
print(f'SB3:            {sb3.__version__}')
print(f'sb3-contrib:    {sb3_contrib.__version__}')
print(f'Optuna:         {optuna.__version__}')
print('\nAll dependencies installed ✓')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8

2026-04-13 13:19:43.358965: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776086383.588930      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776086383.655981      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776086384.201991      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776086384.202049      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776086384.202052      55 computation_placer.cc:177] computation placer alr

SB3:            2.8.0
sb3-contrib:    2.8.0
Optuna:         4.8.0

All dependencies installed ✓


## Step 2: Generate Test Results Dataset

100K chips × 200 tests (Kaggle scale). 70% defect rate across 14 defect types.
Production scale: 1M chips × 1000 tests on AMD EPYC cluster.
> **Note**: The RL environment generates chip profiles on-the-fly from the test catalog.
> This dataset is used for statistical validation and defect rate verification.

In [2]:
from generate_test_data import generate_dataset
import json

summary = generate_dataset(
    output_dir='data',
    n_chips=100_000,
    n_tests=200,
    defect_rate=0.70,
    seed=42,
    chunk_size=25_000,
    fmt='parquet',
)
print('Dataset Summary:')
print(json.dumps(summary, indent=2))


  Generated chips 0–24,999
  Generated chips 25,000–49,999
  Generated chips 50,000–74,999
  Generated chips 75,000–99,999

Dataset: 100,000 chips × 200 tests in 24.9s
Dataset Summary:
{
  "n_chips": 100000,
  "n_tests": 200,
  "defect_rate": 0.7,
  "defect_counts": {
    "scan_fail": 5027,
    "setup_violation": 5004,
    "clock_jitter": 5105,
    "io_fail": 5118,
    "current_leak": 5034,
    "voltage_droop": 5076,
    "no_defect": 29907,
    "memory_fail": 4924,
    "logic_fail": 4941,
    "analog_drift": 4961,
    "hold_violation": 4957,
    "frequency_fail": 5117,
    "electromigration": 5025,
    "power_thermal": 4917,
    "esd_damage": 4887
  },
  "format": "parquet",
  "generation_time_sec": 24.9,
  "file_size_mb": 3.61
}


## Step 3: Environment Validation

Verify the Gymnasium environment scales correctly at 10, 50, and 200 tests.
**Vectorised hot paths**: `_obs()` and `action_masks()` use numpy ops (no Python loops) for 20-50x speedup.

In [3]:
import numpy as np, time
from src.environment import TestFlowEnv, DEFECT_CATEGORIES, CATEGORY_GROUPS

print('Environment Validation')
print('=' * 60)

for n in [10, 50, 200]:
    env = TestFlowEnv(n_tests=n, cost_budget=50.0, time_budget=120.0)
    obs, _ = env.reset(seed=42)
    masks = env.action_masks()
    print(f'n_tests={n:4d}: obs={obs.shape[0]}d  actions={env.action_space.n}  valid={int(masks.sum())}')

# Benchmark environment speed
print('\n=== Environment Speed Benchmark ===')
env200 = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)
obs, _ = env200.reset(seed=42)
N_BENCH = 10_000
t0 = time.perf_counter()
for _ in range(N_BENCH):
    mask = env200.action_masks()
    valid = np.where(mask)[0]
    action = int(np.random.choice(valid))
    obs, r, done, trunc, info = env200.step(action)
    if done or trunc:
        obs, _ = env200.reset()
elapsed = time.perf_counter() - t0
print(f'{N_BENCH:,} steps in {elapsed:.2f}s  →  {N_BENCH/elapsed:.0f} steps/sec')
print(f'Estimated time for 200K training steps: {200_000/(N_BENCH/elapsed)/60:.1f} min')
print('\nEnvironment checks passed ✓')


Environment Validation
n_tests=  10: obs=53d  actions=11  valid=11
n_tests=  50: obs=253d  actions=51  valid=51
n_tests= 200: obs=1003d  actions=201  valid=201

=== Environment Speed Benchmark ===
10,000 steps in 0.51s  →  19767 steps/sec
Estimated time for 200K training steps: 0.2 min

Environment checks passed ✓


## Step 4: Baseline Evaluation

Three heuristic policies evaluated over 500 episodes. RL agents must beat these to be useful.

| Baseline | Strategy |
|----------|----------|
| Random | Uniformly random from valid tests |
| Greedy Coverage | Always highest-coverage test |
| Cost Efficient | Best coverage/cost ratio up to 40% budget |

In [4]:
import pandas as pd
from src.agent import BASELINES, evaluate_policy

env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)

print('Baseline Evaluation (500 episodes each)')
print('=' * 70)
print(f'{"Policy":20s} {"Reward":>10s} {"Accuracy":>10s} {"Cost":>10s} {"Tests":>10s}')
print('-' * 70)

baseline_results = {}
for name, policy_fn in BASELINES.items():
    metrics = evaluate_policy(env, policy_fn, n_episodes=500)
    baseline_results[name] = metrics
    print(f'{name:20s} {metrics["mean_reward"]:>+10.2f} {metrics["accuracy"]:>10.3f} '
          f'{metrics["mean_cost"]:>10.2f} {metrics["mean_tests"]:>10.1f}')

print('=' * 70)
baseline_df = pd.DataFrame(baseline_results).T
best_bl = baseline_df['mean_reward'].max()
print(f'Best baseline: {baseline_df["mean_reward"].idxmax()} (reward={best_bl:+.2f})')
print(f'\nRL must beat: {best_bl:+.2f} reward')


Baseline Evaluation (500 episodes each)
Policy                   Reward   Accuracy       Cost      Tests
----------------------------------------------------------------------
random                    +3.96      0.888      48.70        6.0
greedy_coverage           +3.85      0.886      49.85        7.0
cost_efficient            +5.39      0.876      21.63       11.0
Best baseline: cost_efficient (reward=+5.39)

RL must beat: +5.39 reward


## Step 5: Stage-1 — Train All 4 Algorithms (200K steps each)

**Execution order**: A2C → DQN → PPO → MaskablePPO (fastest to slowest).
Progress prints every 10K steps with elapsed time and ETA.

| Algorithm | Key Setting | Expected Time |
|-----------|------------|---------------|
| A2C | n_steps=5, fast updates | ~1-2h |
| DQN | buffer=100K, replay | ~1-2h |
| PPO | n_steps=2048, clipped | ~1-2h |
| MaskablePPO | action masking | ~2-3h |

> Checkpoints saved to `/kaggle/working/rl_stage1/` every 5K steps via EvalCallback.

In [ ]:
import time as _time, json
import pandas as pd
from pathlib import Path
from src.agent import ALGO_REGISTRY, evaluate_trained_model

STAGE1_STEPS = 200_000
SAVE_DIR = Path('/kaggle/working/rl_stage1')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

stage1_results = {}

print(f'Stage-1: Training 4 algorithms × {STAGE1_STEPS:,} steps')
print('=' * 70)

for algo_name, train_fn in ALGO_REGISTRY.items():
    print(f'\n>>> {algo_name.upper()}')
    env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)
    t0 = _time.time()
    model = train_fn(
        env,
        total_timesteps=STAGE1_STEPS,
        output_dir=str(SAVE_DIR / algo_name),
    )
    train_time = _time.time() - t0
    metrics = evaluate_trained_model(env, model, n_episodes=500)
    metrics['train_time_sec'] = round(train_time, 1)
    stage1_results[algo_name] = metrics
    print(f'  DONE: reward={metrics["mean_reward"]:+.2f} | '
        f'accuracy={metrics["accuracy"]:.3f} | '
        f'cost={metrics["mean_cost"]:.2f} | '
        f'time={train_time/60:.1f}m')

# Save results
stage1_df = pd.DataFrame(stage1_results).T
ranked = stage1_df.sort_values('mean_reward', ascending=False)
top2_algos = list(ranked.index[:2])

print(f'\n{"="*70}')
print('STAGE-1 SUMMARY')
print(stage1_df.to_string())
print(f'\nTop-2 for Stage-2: {top2_algos}')

for algo, row in stage1_df.iterrows():
    imp = row['mean_reward'] - best_bl
    pct = (imp / abs(best_bl)) * 100 if best_bl != 0 else 0
    print(f'  {algo} vs best baseline: {imp:+.2f} ({pct:+.1f}%)')


Stage-1: Training 4 algorithms × 200,000 steps

>>> A2C
Using cuda device
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


  [A2C] Training started — 200,000 steps
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 5.5      |
|    ep_rew_mean        | -0.18    |
| time/                 |          |
|    fps                | 198      |
|    iterations         | 100      |
|    time_elapsed       | 2        |
|    total_timesteps    | 500      |
| train/                |          |
|    entropy_loss       | -3.39    |
|    explained_variance | -0.0013  |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | -75.4    |
|    value_loss         | 454      |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 5.29     |
|    ep_rew_mean        | 2.53     |
| time/                 |          |
|    fps                | 251      |
|    iterations         | 200      |
|    time_elapsed       | 3        |
|    total_timesteps    | 1000    

## Save Outputs for NB2

All models already saved to `/kaggle/working/rl_stage1/` by EvalCallback.
Now save the metrics JSON and create the summary.

> **After this cell**: Output tab → Create Dataset → name it **`rl-stage1-results`**
> Add it as input to NB2.

In [ ]:
import json, shutil
from pathlib import Path

save_data = {
    'baselines': baseline_results,
    'stage1':    stage1_results,
    'top2_algos': top2_algos,
    'best_base_reward': float(best_bl),
}

with open(SAVE_DIR / 'stage1_results.json', 'w') as f:
    json.dump(save_data, f, indent=2)

print('=== NB1 ARTIFACTS ===')
for f in sorted(SAVE_DIR.rglob('*')):
    if f.is_file():
        print(f'  {str(f.relative_to(SAVE_DIR)):60s} {f.stat().st_size/1e3:6.0f} KB')

total = sum(f.stat().st_size for f in SAVE_DIR.rglob('*') if f.is_file())
print(f'\nTotal: {total/1e6:.1f} MB')
print('\nNB1 complete!')
print('Next: Output tab → Create Dataset → name it rl-stage1-results')
print('Then open NB2 and add rl-stage1-results as input dataset')
